# Spectogram-based Convolutional Neural Network for Spoken Digit Recognition

Waleed Jamil, Syracuse University

# Import Libraries and .wav data



Mounting dataset from Google Drive.

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import json
import scipy.signal as signal
import numpy as np
from scipy.io import wavfile
import os
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.layers import Input

In [ ]:
# Path to your folder of wav files
folder_path = "/content/drive/MyDrive/Free_Spoken_Digit_Recordings"

# List WAV files


In [ ]:
wav_files = [f for f in os.listdir(folder_path) if f.endswith('.wav')]

print(f"Found {len(wav_files)} WAV files")
print(wav_files[:5])

Found 3000 WAV files
['5_lucas_4.wav', '0_lucas_0.wav', '0_nicolas_20.wav', '5_yweweler_40.wav', '3_nicolas_28.wav']


Load Audio Data

In [ ]:
audio_data_list = []

for filename in wav_files:
  file_path = os.path.join(folder_path, filename)
  try:
    sampling_rate, audio_samples = wavfile.read(file_path)
    audio_data_list.append({
        "filename": filename,
        "sampling_rate": sampling_rate,
        "audio_samples": audio_samples.tolist()
    })

  except Exception as e:
      print(f"Error processing {filename}: {e}")

print(f"Processed {len(audio_data_list)} out of {len(wav_files)} WAV files.")
print(f"First entry in audio_data_list: {audio_data_list[0]['filename']}, Sampling Rate: {audio_data_list[0]['sampling_rate']} Hz, Audio Samples Length: {len(audio_data_list[0]['audio_samples'])}")

Processed 3000 out of 3000 WAV files.
First entry in audio_data_list: 5_lucas_4.wav, Sampling Rate: 8000 Hz, Audio Samples Length: 4067


# Write Audio Data into a JSON


In [ ]:
output_json_path = 'audio_data.json'

with open(output_json_path, 'w') as f:
    json.dump(audio_data_list, f, indent=4)

print(f"Audio data successfully written to {output_json_path}")

Audio data successfully written to audio_data.json


In [ ]:
with open("audio_data.json", 'r', encoding='utf-8') as f:
    data = json.load(f)

In [ ]:
audio_df = pd.DataFrame.from_dict(data)
audio_df.head()

,filename,sampling_rate,audio_samples
0,5_lucas_4.wav,8000,"[-22, -5, -23, -21, 0, -3, -4, -20, -6, -2, 23..."
1,0_lucas_0.wav,8000,"[13, 4, -5, 8, 5, 7, -1, 3, 7, 6, -1, -4, 9, 4..."
2,0_nicolas_20.wav,8000,"[0, -256, 0, -256, 0, 0, -256, 0, -256, -256, ..."
3,5_yweweler_40.wav,8000,"[3, -2, 19, -10, 5, -6, -26, -19, -6, 3, 21, -..."
4,3_nicolas_28.wav,8000,"[0, 0, -256, 0, -256, 0, -256, 0, -256, -256, ..."


In [ ]:
audio_df.describe()

,sampling_rate
count,3000.0
mean,8000.0
std,0.0
min,8000.0
25%,8000.0
50%,8000.0
75%,8000.0
max,8000.0


In [ ]:
print(f"Data type of 'audio_samples' column: {audio_df['audio_samples'].dtype}")

print(audio_df['audio_samples'].head())

Data type of 'audio_samples' column: object
0    [-22, -5, -23, -21, 0, -3, -4, -20, -6, -2, 23...
1    [13, 4, -5, 8, 5, 7, -1, 3, 7, 6, -1, -4, 9, 4...
2    [0, -256, 0, -256, 0, 0, -256, 0, -256, -256, ...
3    [3, -2, 19, -10, 5, -6, -26, -19, -6, 3, 21, -...
4    [0, 0, -256, 0, -256, 0, -256, 0, -256, -256, ...
Name: audio_samples, dtype: object


Convert Audio Samples to NumPy Arrays because libraries like SciPy used to create spectograms are optimized for numpy arrays

In [ ]:
audio_df['audio_samples'] = audio_df['audio_samples'].apply(lambda x: np.array(x))

type(audio_df['audio_samples'].iloc[0])

numpy.ndarray

# Feature Engineering: Generate Spectrograms


Raw audio waveforms are 1‑D signals -> not ideal for CNNs as they do not excel at learning from them

Spectrograms turn audio into 2‑D images

audio_samples (np.ndarray): Array of audio samples.

sampling_rate (int): Sampling rate of the audio.
    
nperseg (int): Length of each segment for the spectrogram.
    
noverlap (int): Number of points to overlap between segments.

Returns: Spectrogram in decibel scale.


In [ ]:
def generate_spectrogram(audio_samples, sampling_rate, nperseg=256, noverlap=128):
  # Computing the spectrogram
  frequencies, segment_times, stft = signal.spectrogram(audio_samples,
                                                        fs=sampling_rate,
                                                        nperseg=nperseg,
                                                        noverlap=noverlap)

  # Converting magnitude to decibel scale
  magnitude_spectrogram = np.abs(stft)

  # Adding constant to avoid log(0)
  spectrogram_db = 10 * np.log10(magnitude_spectrogram + 1e-10)

  return spectrogram_db

In [ ]:
# Applying the function to create the 'spectrogram' column
audio_df['spectrogram'] = audio_df.apply(lambda row: generate_spectrogram(
    row['audio_samples'], row['sampling_rate']), axis=1)

audio_df.head()

,filename,sampling_rate,audio_samples,spectrogram
0,5_lucas_4.wav,8000,"[-22, -5, -23, -21, 0, -3, -4, -20, -6, -2, 23...","[[-20.981280678931917, -29.82168844943803, -20..."
1,0_lucas_0.wav,8000,"[13, 4, -5, 8, 5, 7, -1, 3, 7, 6, -1, -4, 9, 4...","[[-24.634188951252543, -18.43509497255891, -35..."
2,0_nicolas_20.wav,8000,"[0, -256, 0, -256, 0, 0, -256, 0, -256, -256, ...","[[5.752543259891176, 12.98499805307281, 21.700..."
3,5_yweweler_40.wav,8000,"[3, -2, 19, -10, 5, -6, -26, -19, -6, 3, 21, -...","[[8.497088490540332, 21.880516359356655, 17.55..."
4,3_nicolas_28.wav,8000,"[0, 0, -256, 0, -256, 0, -256, 0, -256, -256, ...","[[16.351736487444338, 12.18812151738127, 28.19..."


In [ ]:
# Verifying the spectogram shape
if not audio_df.empty:
  print(f"Shape of the first generated spectrogram: {audio_df['spectrogram'].iloc[0].shape}")


Shape of the first generated spectrogram: (129, 30)


# Preparation of data for CNN

Creating a new label column by extracting the filename and seperating at the underscore

In [ ]:
audio_df['label'] = audio_df['filename'].apply(lambda x: int(x.split('_')[0]))

print(audio_df[['filename', 'label']].head())

            filename  label
0      5_lucas_4.wav      5
1      0_lucas_0.wav      0
2   0_nicolas_20.wav      0
3  5_yweweler_40.wav      5
4   3_nicolas_28.wav      3


Calculating max height and width of spectograms for padding

In [ ]:
max_height = 0
max_width = 0

for spectrogram in audio_df['spectrogram']:
  if spectrogram.shape[0] > max_height:
    max_height = spectrogram.shape[0]

  if spectrogram.shape[1] > max_width:
    max_width = spectrogram.shape[1]

print(f"Maximum spectrogram height: {max_height}")
print(f"Maximum spectrogram width: {max_width}")

Maximum spectrogram height: 129
Maximum spectrogram width: 141


To ensure consistent shape, max height and width will be used to pad or truncate the spectograms

In [ ]:
def pad_spectrogram(spectrogram, target_height, target_width):
  height, width = spectrogram.shape

  # Error check in case height and width found is larger than max
  # Truncate if larger -> these conditions should never be met
  if height > target_height:
      spectrogram = spectrogram[:target_height, :]
  if width > target_width:
      spectrogram = spectrogram[:, :target_width]

  # Pad if smaller
  padded_spectrogram = np.pad(spectrogram, ((0, max(0, target_height - height)),
                                            (0, max(0, target_width - width))),
                              mode='constant', constant_values=0)
  return padded_spectrogram

Ensuring all spectograms have consistent shape by applying padding functino to 'spectogram' column of audio_df


In [ ]:
audio_df['spectrogram'] = audio_df['spectrogram'].apply(lambda x: pad_spectrogram(x, max_height, max_width))
print(f"First spectrogram shape after padding/truncation: {audio_df['spectrogram'].iloc[0].shape}")

First spectrogram shape after padding/truncation: (129, 141)


Converting spectogram into a single array with added channel dimension to make CNN input

shape[0] = number of samples

shape[1] = spectogram height

shape[2] = spectogram width

1 = number of channels (new dimension)

In [ ]:
spectrograms = np.stack(audio_df['spectrogram'].values)

spectrograms = spectrograms.reshape(spectrograms.shape[0], spectrograms.shape[1], spectrograms.shape[2], 1)

spectrograms.shape

(3000, 129, 141, 1)

Converting label column to array for training and splitting

In [ ]:
labels = np.array(audio_df['label'].values)

labels.shape

(3000,)

Splitting processed spectograms into training and testing dataset

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(spectrograms, labels, test_size=0.2, random_state=24)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

Shape of X_train: (2400, 129, 141, 1)
Shape of X_test: (600, 129, 141, 1)
Shape of y_train: (2400,)
Shape of y_test: (600,)


# Build the CNN Model

Create Convolutional Neural Network structure

In [ ]:
# Defining the input shape for the CNN:
# height and width of spectogram and 1 channel
input_shape = (max_height, max_width, 1)

# Number of classes = amount of labels (digits 0 - 9)
num_classes = len(np.unique(labels))

# Initializing sequential model
model = Sequential()

# Addding input layer
model.add(Input(shape=input_shape))

# Convolutional with relu activation and Pooling layers
model.add(Conv2D(32, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))

# Additional convolutional and pooling layers
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))

model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))

# Flattening output for dense layers
model.add(Flatten())

# Adding dense layers with relu activation
model.add(Dense(128, activation='relu'))
model.add(Dense(64, activation='relu'))

# Output layer with softmax activation
# Given this is a multi-class classification (digits 0-9)
model.add(Dense(num_classes, activation='softmax'))

# Summarizing model architecture
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 127, 139, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 69, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 67, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 33, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 31, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 15, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 26880)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     3,440,768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,542,346 (13.51 MB)

 Trainable params: 3,542,346 (13.51 MB)

 Non-trainable params: 0 (0.00 B)

# Compile and Train the CNN Model

Configuring CNN then training on training dataset

In [ ]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history = model.fit(X_train, y_train, epochs=5, validation_data=(X_test, y_test))


Epoch 1/5
21/75 ━━━━━━━━━━━━━━━━━━━━ 1:16 1s/step - accuracy: 0.1205 - loss: 2.8566

# Evaluate the CNN Model


Evaluating CNN performance on test dataset

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)

print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")